# 02 - Journal Finder Model

Bu notebook'ta amaç, kullanıcının girdiği makale özetine göre en uygun 5 akademik dergiyi önermektir.

Modelleme yaklaşımı:

- Abstract metinleri temizlenir.
- TF-IDF ile sayısal vektörlere dönüştürülür.
- Lineer metin sınıflandırma modeli eğitilir.
- Model, her journal için skor üretir.
- En yüksek skora sahip ilk 5 journal önerilir.


In [12]:
from pathlib import Path
import re
import html
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, top_k_accuracy_score, classification_report

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 250)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "master_articles.csv"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)
print("Data exists:", DATA_PATH.exists())


Project root: /Users/alp/dev/journal-finder-project
Data path: /Users/alp/dev/journal-finder-project/data/master_articles.csv
Data exists: True


In [13]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Unique journals:", df["journal_name"].nunique())

df.head()


Shape: (22655, 19)
Unique journals: 454


,academic_record_id,wos_uid,title,abstract_text,publication_id,journal_name,journal_abbreviation,issn,pub_year,cite_count,record_impact_factor,q_value,document_type,language,publication_type,author_keywords,keyword_plus,subjects,abstract_word_count
0,88652,WOS:000166979500001,An updated survey of GA-based multiobjective optimization techniques,"<p>After using evolutionary techniques for single-objective optimization during more than two decades, the incorporation of more than one objective in the fitness function has finally become a popular area of research. As a consequence, many new ...",11050,ACM COMPUTING SURVEYS,ACM COMPUT SURV,0360-0300,2000,337,NaN,NaN,Review,English,Journal,algorithms; artificial intelligence; genetic algorithms; multicriteria optimization; multiobjective optimization; vector optimization,GENETIC ALGORITHM; MULTICRITERIA OPTIMIZATION; STRUCTURAL OPTIMIZATION; ENGINEERING DESIGN; SEARCH; SOLVE,"Computer Science, Theory & Methods; Computer Science",153
1,88653,WOS:000168229900003,The state of the art in distributed query processing,"<p>Distributed data processing is becoming a reality. Businesses want to do it for many reasons, and they often must do it in order to stay competitive. While much of the infrastructure for distributed data processing is already there (e.g., mode...",11050,ACM COMPUTING SURVEYS,ACM COMPUT SURV,0360-0300,2000,298,NaN,NaN,Review,English,Journal,query optimization; query execution; client-server databases; middleware; multitier architectures; database application systems; wrappers; replication; caching; economic models for query processing; dissemination-based information systems,DATABASE-SYSTEMS; DATA REPLICATION; PERFORMANCE; JOIN; OPTIMIZATION; ALGORITHMS; ACCESS,"Computer Science, Theory & Methods; Computer Science",219
2,88654,WOS:000168229900001,Logical models of argument,<p>Logical models of argument formalize commonsense reasoning while taking process and computation seriously. This survey discusses the main ideas that characterize different logical models of argument. It presents the formal features of a few ma...,11050,ACM COMPUTING SURVEYS,ACM COMPUT SURV,0360-0300,2000,238,NaN,NaN,Review,English,Journal,defeasible argumentation; argumentative systems; defeasible reasoning,IMPLEMENTATION; FRAMEWORK,"Computer Science, Theory & Methods; Computer Science",100
3,88655,WOS:000166979500002,Information retrieval on the Web,"<p>In this paper we review studies of the growth of the Internet and technologies that are useful for information search and retrieval on the Web. We present data an the Internet from several different sources, e.g., current as well as projected ...",11050,ACM COMPUTING SURVEYS,ACM COMPUT SURV,0360-0300,2000,220,NaN,NaN,Review,English,Journal,algorithms; theory; clustering; indexing; information retrieval; Internet; knowledge management; search engine; World Wide Web,WORLD-WIDE-WEB; SEARCH; SYSTEM; DIRECTIONS; ENGINE; QUERY,"Computer Science, Theory & Methods; Computer Science",160
4,88656,WOS:000168987700002,A guided tour to approximate string matching,<p>We survey the current techniques to cope with the problem of string matching that allows errors. This is becoming a more and more relevant issue for many fast growing areas such as information retrieval and computational biology. We focus on o...,11050,ACM COMPUTING SURVEYS,ACM COMPUT SURV,0360-0300,2001,858,NaN,NaN,Review,English,Journal,algorithms; edit distance; Levenshtein distance; online string matching; text searching allowing errors,FAST ALGORITHMS; SUBQUADRATIC ALGORITHM; COMMON ANCESTORS; TEXT; MISMATCHES; SEARCH; CONSTRUCTION; HYPERTEXT; DISTANCES,"Computer Science, Theory & Methods; Computer Science",103


## Text Cleaning

Veritabanındaki abstract metinlerinde `<p>` gibi HTML tagleri var. Bunları temizliyoruz.


In [14]:
def clean_text(text):
    text = "" if pd.isna(text) else str(text)
    text = html.unescape(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\-\+\#\.\,\;\:\(\)\[\]\/ ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["abstract_clean"] = df["abstract_text"].apply(clean_text)
df["title_clean"] = df["title"].apply(clean_text)

df[["title", "abstract_text", "abstract_clean"]].head()


,title,abstract_text,abstract_clean
0,An updated survey of GA-based multiobjective optimization techniques,"<p>After using evolutionary techniques for single-objective optimization during more than two decades, the incorporation of more than one objective in the fitness function has finally become a popular area of research. As a consequence, many new ...","After using evolutionary techniques for single-objective optimization during more than two decades, the incorporation of more than one objective in the fitness function has finally become a popular area of research. As a consequence, many new evo..."
1,The state of the art in distributed query processing,"<p>Distributed data processing is becoming a reality. Businesses want to do it for many reasons, and they often must do it in order to stay competitive. While much of the infrastructure for distributed data processing is already there (e.g., mode...","Distributed data processing is becoming a reality. Businesses want to do it for many reasons, and they often must do it in order to stay competitive. While much of the infrastructure for distributed data processing is already there (e.g., modern ..."
2,Logical models of argument,<p>Logical models of argument formalize commonsense reasoning while taking process and computation seriously. This survey discusses the main ideas that characterize different logical models of argument. It presents the formal features of a few ma...,Logical models of argument formalize commonsense reasoning while taking process and computation seriously. This survey discusses the main ideas that characterize different logical models of argument. It presents the formal features of a few main ...
3,Information retrieval on the Web,"<p>In this paper we review studies of the growth of the Internet and technologies that are useful for information search and retrieval on the Web. We present data an the Internet from several different sources, e.g., current as well as projected ...","In this paper we review studies of the growth of the Internet and technologies that are useful for information search and retrieval on the Web. We present data an the Internet from several different sources, e.g., current as well as projected num..."
4,A guided tour to approximate string matching,<p>We survey the current techniques to cope with the problem of string matching that allows errors. This is becoming a more and more relevant issue for many fast growing areas such as information retrieval and computational biology. We focus on o...,We survey the current techniques to cope with the problem of string matching that allows errors. This is becoming a more and more relevant issue for many fast growing areas such as information retrieval and computational biology. We focus on onli...


In [15]:
df_model = df.copy()

df_model = df_model.dropna(subset=["journal_name", "abstract_clean"])
df_model = df_model[df_model["abstract_clean"].str.split().str.len() >= 50]

journal_counts = df_model["journal_name"].value_counts()

print("Before journal count filtering")
print("Rows:", len(df_model))
print("Journals:", df_model["journal_name"].nunique())
print("Min article count per journal:", journal_counts.min())

# Stratified train/test split için çok az örnekli journal varsa çıkarmak daha güvenli.
MIN_ARTICLES_PER_JOURNAL = 5

valid_journals = journal_counts[journal_counts >= MIN_ARTICLES_PER_JOURNAL].index
df_model = df_model[df_model["journal_name"].isin(valid_journals)].copy()

print("\nAfter journal count filtering")
print("Rows:", len(df_model))
print("Journals:", df_model["journal_name"].nunique())
print("Min article count per journal:", df_model["journal_name"].value_counts().min())

df_model["journal_name"].value_counts().describe()


Before journal count filtering
Rows: 22637
Journals: 454
Min article count per journal: 1

After journal count filtering
Rows: 22547
Journals: 402
Min article count per journal: 5


count    402.000000
mean      56.087065
std       20.179614
min        5.000000
25%       43.000000
50%       60.000000
75%       75.000000
max       76.000000
Name: count, dtype: float64

## Train / Test Split

Model sadece abstract üzerinden eğitiliyor. Çünkü proje sonunda kullanıcıdan sadece abstract alınacak.


In [16]:
X = df_model["abstract_clean"]
y = df_model["journal_name"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))
print("Train journals:", y_train.nunique())
print("Test journals:", y_test.nunique())


Train size: 18037
Test size: 4510
Train journals: 402
Test journals: 402


## Model Training

Burada TF-IDF + lineer sınıflandırıcı kullanıyoruz.

`SGDClassifier(loss="log_loss")`, çok sınıflı metin sınıflandırması için hızlı ve uygundur. Ayrıca `predict_proba` desteklediği için top-5 journal skoru çıkarabiliriz.


In [17]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.90,
        max_features=50000,
        sublinear_tf=True
    )),
    ("clf", SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=1e-5,
        max_iter=1000,
        tol=1e-3,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)

print("Model training completed.")


Model training completed.


## Evaluation

Top-1 accuracy: Modelin ilk tahmini doğru mu?

Top-5 accuracy: Modelin önerdiği ilk 5 journal içinde gerçek journal var mı?

Bu proje için asıl önemli metrik **Top-5 Accuracy**, çünkü sistem kullanıcıya 5 journal öneriyor.


In [18]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

top1_acc = accuracy_score(y_test, y_pred)
top5_acc = top_k_accuracy_score(
    y_test,
    y_proba,
    k=5,
    labels=model.named_steps["clf"].classes_
)

print("Top-1 Accuracy:", round(top1_acc, 4))
print("Top-5 Accuracy:", round(top5_acc, 4))


Top-1 Accuracy: 0.3623
Top-5 Accuracy: 0.6616


In [19]:
print(classification_report(y_test, y_pred, zero_division=0))


                                                                                                                precision    recall  f1-score   support

                                                                                         ACM COMPUTING SURVEYS       0.15      0.13      0.14        15
                                                                ACM JOURNAL ON COMPUTING AND CULTURAL HERITAGE       0.80      0.80      0.80         5
                                                     ACM JOURNAL ON EMERGING TECHNOLOGIES IN COMPUTING SYSTEMS       0.42      0.50      0.45        10
                                                                     ACM SIGCOMM COMPUTER COMMUNICATION REVIEW       0.26      0.47      0.33        15
                                                                                ACM TRANSACTIONS ON ALGORITHMS       0.15      0.33      0.21         9
                                                                        ACM TRANSACTION

## Top-5 Journal Recommendation Function

Bu fonksiyon final projenin ana çıktısıdır.

Kullanıcı abstract girer, sistem en alakalı 5 journal'ı listeler.


In [20]:
def recommend_top5_journals(abstract_text, model, top_n=5):
    cleaned = clean_text(abstract_text)

    probabilities = model.predict_proba([cleaned])[0]
    classes = model.named_steps["clf"].classes_

    top_indices = np.argsort(probabilities)[::-1][:top_n]

    result = pd.DataFrame({
        "rank": range(1, top_n + 1),
        "journal_name": classes[top_indices],
        "score": probabilities[top_indices]
    })

    return result

sample_abstract = X_test.iloc[0]
actual_journal = y_test.iloc[0]

print("Actual journal:")
print(actual_journal)

recommend_top5_journals(sample_abstract, model)


Actual journal:
COMPUTERS & GEOSCIENCES


,rank,journal_name,score
0,1,COMPUTERS & GEOSCIENCES,0.024251
1,2,COMPUTER METHODS AND PROGRAMS IN BIOMEDICINE,0.013152
2,3,ADVANCES IN ENGINEERING SOFTWARE,0.012052
3,4,COMPUTER STANDARDS & INTERFACES,0.011154
4,5,JOURNAL OF VISUAL LANGUAGES AND COMPUTING,0.010870


## Manual Test

Aşağıdaki örnek abstract değiştirilebilir.


In [21]:
manual_abstract = """
This study proposes a machine learning based approach for text classification and information retrieval.
The proposed model uses natural language processing techniques to represent documents and classify them
according to their semantic content. Experimental results show that the method improves classification
performance on benchmark datasets.
"""

recommend_top5_journals(manual_abstract, model)


,rank,journal_name,score
0,1,INFORMATION RETRIEVAL,0.058033
1,2,JOURNAL OF INTELLIGENT INFORMATION SYSTEMS,0.043727
2,3,ACM TRANSACTIONS ON INFORMATION SYSTEMS,0.042804
3,4,FOUNDATIONS AND TRENDS IN INFORMATION RETRIEVAL,0.027433
4,5,MALAYSIAN JOURNAL OF COMPUTER SCIENCE,0.025765


## Save Model

Model `.pkl` olarak kaydedilir. Daha sonra ayrı bir Python dosyasından veya basit bir arayüzden kullanılabilir.


In [22]:
model_path = MODEL_DIR / "journal_recommender_tfidf_sgd.pkl"
metadata_path = MODEL_DIR / "journal_recommender_metadata.json"

joblib.dump(model, model_path)

metadata = {
    "model_name": "TF-IDF + SGDClassifier(log_loss)",
    "rows_used": int(len(df_model)),
    "unique_journals": int(df_model["journal_name"].nunique()),
    "min_articles_per_journal": int(MIN_ARTICLES_PER_JOURNAL),
    "top1_accuracy": float(top1_acc),
    "top5_accuracy": float(top5_acc),
    "features": "abstract_clean only"
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)

model_data_path = PROJECT_ROOT / "data" / "model_articles.csv"
df_model.to_csv(model_data_path, index=False)

print("Saved model:", model_path)
print("Saved metadata:", metadata_path)
print("Saved model dataset:", model_data_path)


Saved model: /Users/alp/dev/journal-finder-project/models/journal_recommender_tfidf_sgd.pkl
Saved metadata: /Users/alp/dev/journal-finder-project/models/journal_recommender_metadata.json
Saved model dataset: /Users/alp/dev/journal-finder-project/data/model_articles.csv
